# Two-section Figure 4 rescue heatmap

This notebook rebuilds the two rescue heatmaps from the original count tables and separates the rows into **autosomal neural** and **autosomal skeletal muscle** sections. The old WT/KO/DKO and new WT/KO/KO+X1 experiments are normalized and row-scaled separately, exactly as in the existing workflow.

## 1. Configuration

Paths, scientific thresholds, gene lists, labels, colors, and figure dimensions are kept here so they can be edited without searching through the analysis code.

In [ ]:
count_dir <- "source_files"
output_dir <- file.path("generated_figures", "Fig4H")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

output_file <- file.path(
  output_dir,
  "Fig4H_autosomal_neural_skeletal_rescue_heatmaps.tiff"
)

# Scientific rescue thresholds retained from the original Figure 4H analysis.
padj_cutoff <- 0.05
log2fc_cutoff <- 0.5
minimum_total_count <- 10

# Row order in the revised figure.
autosomal_neural_genes <- c(
  "PKNOX2", "TTYH2", "RBFOX1", "RNF165", "SORCS2", "NAV1"
)

autosomal_skeletal_muscle_genes <- c(
  "PAX7", "CDON", "BOC", "GAS1", "EPHB1"
)

original_nine_XIST_binding_rescue_genes <- c(
  "EPHB1", "CDON", "PKNOX2", "TTYH2", "RBFOX1",
  "PAX7", "RNF165", "SORCS2", "NAV1"
)
new_skeletal_rescue_genes <- c("BOC", "GAS1")

heatmap_genes <- c(
  autosomal_neural_genes,
  autosomal_skeletal_muscle_genes
)
row_gaps <- length(autosomal_neural_genes)

# Figure-aesthetic settings.
heatmap_colors <- c(
  "#3E86B8", "#5C9BC8", "#7CAFD6", "#A0C5E2", "#C6DDEC",
  "#F4E6AC",
  "#F0C79F", "#EBA486", "#E4826E", "#D66358", "#C54843"
)
cell_width <- 30
cell_height <- 30
figure_width <- 12
figure_height <- 7
figure_dpi <- 600
row_font_size <- 18
column_font_size <- 18


## 2. Packages

In [ ]:
suppressPackageStartupMessages({
  library(DESeq2)
  library(pheatmap)
  library(grid)
})

## 3. Read and verify the count tables

The two experiments each contain two WT, two AFF3 KO, and two rescue samples.

In [ ]:
old_sample_files <- c(
  old_WT_1 = "AFF3_F_WT_1_mm_counts_revised.txt",
  old_WT_2 = "AFF3_F_WT_2_mm_counts_revised.txt",
  old_KO_1 = "AFF3_F_KO_1_mm_counts_revised.txt",
  old_KO_2 = "AFF3_F_KO_2_mm_counts_revised.txt",
  old_DKO_1 = "AFF3_F_KO_Xt_KO_A3_1_mm_counts_revised.txt",
  old_DKO_2 = "AFF3_F_KO_Xt_KO_A3_2_mm_counts_revised.txt"
)

new_sample_files <- c(
  new_WT_1 = "WT_1_mm_counts_revised.txt",
  new_WT_2 = "WT_2_mm_counts_revised.txt",
  new_KO_1 = "KO_1_mm_counts_revised.txt",
  new_KO_2 = "KO_2_mm_counts_revised.txt",
  new_KOX1_1 = "KOX1_1_mm_counts_revised.txt",
  new_KOX1_2 = "KOX1_2_mm_counts_revised.txt"
)

build_count_matrix <- function(sample_files) {
  input_paths <- file.path(count_dir, unname(sample_files))
  missing_files <- input_paths[!file.exists(input_paths)]
  if (length(missing_files) > 0) {
    stop("Missing count tables: ", paste(missing_files, collapse = ", "))
  }

  count_tables <- lapply(
    input_paths,
    read.table,
    header = TRUE,
    stringsAsFactors = FALSE
  )

  reference_genes <- count_tables[[1]]$Geneid
  same_gene_order <- vapply(
    count_tables,
    function(count_table) identical(count_table$Geneid, reference_genes),
    logical(1)
  )
  stopifnot(all(same_gene_order))

  count_matrix <- do.call(
    cbind,
    lapply(count_tables, function(count_table) count_table$counts)
  )
  rownames(count_matrix) <- reference_genes
  colnames(count_matrix) <- names(sample_files)
  count_matrix
}

old_count_matrix <- build_count_matrix(old_sample_files)
new_count_matrix <- build_count_matrix(new_sample_files)

stopifnot(
  all(heatmap_genes %in% rownames(old_count_matrix)),
  all(heatmap_genes %in% rownames(new_count_matrix))
)

dim(old_count_matrix)
dim(new_count_matrix)

## 4. Reproduce the original rescue analysis

A gene is rescued only when it is significantly down in KO versus WT and significantly up in the rescue condition versus KO. The same thresholds are applied independently to both experiments.

In [ ]:
run_rescue_deseq <- function(count_matrix, condition_levels) {
  col_data <- data.frame(
    condition = factor(condition_levels, levels = unique(condition_levels)),
    row.names = colnames(count_matrix)
  )

  dds <- DESeqDataSetFromMatrix(
    countData = count_matrix,
    colData = col_data,
    design = ~ condition
  )
  dds <- dds[rowSums(counts(dds)) >= minimum_total_count, ]
  DESeq(dds, quiet = TRUE)
}

genes_down <- function(result_table) {
  rownames(result_table)[
    !is.na(result_table$padj) &
      result_table$padj < padj_cutoff &
      result_table$log2FoldChange < -log2fc_cutoff
  ]
}

genes_up <- function(result_table) {
  rownames(result_table)[
    !is.na(result_table$padj) &
      result_table$padj < padj_cutoff &
      result_table$log2FoldChange > log2fc_cutoff
  ]
}

old_dds <- run_rescue_deseq(
  old_count_matrix,
  c("WT", "WT", "KO", "KO", "DKO", "DKO")
)
new_dds <- run_rescue_deseq(
  new_count_matrix,
  c("WT", "WT", "KO", "KO", "KOX1", "KOX1")
)

old_KO_vs_WT <- results(old_dds, contrast = c("condition", "KO", "WT"))
old_DKO_vs_KO <- results(old_dds, contrast = c("condition", "DKO", "KO"))
new_KO_vs_WT <- results(new_dds, contrast = c("condition", "KO", "WT"))
new_KOX1_vs_KO <- results(new_dds, contrast = c("condition", "KOX1", "KO"))

old_rescued_genes <- intersect(
  genes_down(old_KO_vs_WT),
  genes_up(old_DKO_vs_KO)
)
new_rescued_genes <- intersect(
  genes_down(new_KO_vs_WT),
  genes_up(new_KOX1_vs_KO)
)
shared_rescued_genes <- intersect(old_rescued_genes, new_rescued_genes)

XIST_binding_genes <- unique(
  read.csv(
    file.path(count_dir, "XIST_binding_933_genes.csv"),
    check.names = FALSE
  )$Gene
)
shared_rescued_XIST_binding_genes <- intersect(
  shared_rescued_genes,
  XIST_binding_genes
)

stopifnot(
  all(original_nine_XIST_binding_rescue_genes %in% shared_rescued_XIST_binding_genes),
  all(new_skeletal_rescue_genes %in% shared_rescued_genes),
  !any(new_skeletal_rescue_genes %in% XIST_binding_genes)
)

c(
  old_rescued = length(old_rescued_genes),
  new_rescued = length(new_rescued_genes),
  shared_rescued = length(shared_rescued_genes),
  shared_rescued_XIST_binding = length(shared_rescued_XIST_binding_genes)
)

## 5. Inspect the two added skeletal-muscle rescue genes

This compact audit keeps the scientific selection separate from the heatmap rendering. Negative KO/WT and positive rescue/KO values are the required directions.

In [ ]:
extract_result <- function(result_table, genes, prefix) {
  data.frame(
    Gene = genes,
    setNames(
      data.frame(
        log2FC = result_table[genes, "log2FoldChange"],
        padj = result_table[genes, "padj"]
      ),
      paste0(prefix, c("_log2FC", "_padj"))
    ),
    row.names = NULL,
    check.names = FALSE
  )
}

candidate_audit <- Reduce(
  function(left, right) merge(left, right, by = "Gene", sort = FALSE),
  list(
    extract_result(old_KO_vs_WT, new_skeletal_rescue_genes, "old_KO_vs_WT"),
    extract_result(old_DKO_vs_KO, new_skeletal_rescue_genes, "old_DKO_vs_KO"),
    extract_result(new_KO_vs_WT, new_skeletal_rescue_genes, "new_KO_vs_WT"),
    extract_result(new_KOX1_vs_KO, new_skeletal_rescue_genes, "new_KOX1_vs_KO")
  )
)

candidate_audit <- candidate_audit[
  match(new_skeletal_rescue_genes, candidate_audit$Gene),
  ,
  drop = FALSE
]
candidate_audit

## 6. VST normalization and row z-scores

Each experiment is transformed and standardized independently. This preserves the existing Figure 4 heatmap analysis rather than placing old and new experiments on one shared scale.

In [ ]:
calculate_row_z_scores <- function(dds) {
  vst_matrix <- assay(vst(dds, blind = FALSE))
  z_matrix <- t(scale(t(vst_matrix)))
  z_matrix[complete.cases(z_matrix), , drop = FALSE]
}

old_z_all <- calculate_row_z_scores(old_dds)
new_z_all <- calculate_row_z_scores(new_dds)

old_z_selected <- old_z_all[heatmap_genes, , drop = FALSE]
new_z_selected <- new_z_all[heatmap_genes, , drop = FALSE]

stopifnot(
  identical(rownames(old_z_selected), heatmap_genes),
  identical(rownames(new_z_selected), heatmap_genes),
  !anyNA(old_z_selected),
  !anyNA(new_z_selected)
)

## 7. Preserve the original heatmap color mapping

The color limits are calculated from the original nine autosomal XIST-binding rescue genes, matching the prior combined-heatmap workflow.

In [ ]:
number_of_colors <- length(heatmap_colors)

old_reference_z <- old_z_all[
  original_nine_XIST_binding_rescue_genes,
  ,
  drop = FALSE
]
new_reference_z <- new_z_all[
  original_nine_XIST_binding_rescue_genes,
  ,
  drop = FALSE
]

old_breaks <- seq(
  min(old_reference_z),
  max(old_reference_z),
  length.out = number_of_colors + 1
)
new_breaks <- seq(
  min(new_reference_z),
  max(new_reference_z),
  length.out = number_of_colors + 1
)

convert_to_color_index <- function(values, source_breaks) {
  color_index <- findInterval(values, source_breaks, all.inside = TRUE)
  pmin(pmax(color_index, 1), number_of_colors)
}

to_color_index_matrix <- function(z_matrix, source_breaks) {
  matrix(
    convert_to_color_index(z_matrix, source_breaks),
    nrow = nrow(z_matrix),
    ncol = ncol(z_matrix),
    dimnames = dimnames(z_matrix)
  )
}

old_color_index <- to_color_index_matrix(old_z_selected, old_breaks)
new_color_index <- to_color_index_matrix(new_z_selected, new_breaks)

## 8. Plot the two-section heatmaps

One horizontal gap separates the autosomal neural and autosomal skeletal-muscle sections. Clustering remains disabled and the gene order remains unchanged.

In [ ]:
make_heatmap_panel <- function(
    color_index_matrix, source_breaks, sample_labels, panel_title) {
  legend_z_scores <- c(-1, -0.5, 0, 0.5, 1)
  legend_color_indices <- convert_to_color_index(
    legend_z_scores,
    source_breaks
  )

  pheatmap(
    color_index_matrix,
    cluster_rows = FALSE,
    cluster_cols = FALSE,
    show_rownames = TRUE,
    show_colnames = TRUE,
    labels_col = sample_labels,
    color = heatmap_colors,
    breaks = seq(0.5, number_of_colors + 0.5, by = 1),
    gaps_row = row_gaps,
    legend = TRUE,
    legend_breaks = legend_color_indices,
    legend_labels = c("-1", "-0.5", "0", "0.5", "1"),
    main = panel_title,
    fontsize = 16,
    fontsize_row = row_font_size,
    fontsize_col = column_font_size,
    border_color = "grey60",
    scale = "none",
    cellwidth = cell_width,
    cellheight = cell_height,
    angle_col = "45",
    silent = TRUE
  )
}

old_panel <- make_heatmap_panel(
  old_color_index,
  old_breaks,
  c("WT 1", "WT 2", "KO 1", "KO 2", "DKO 1", "DKO 2"),
  "AFF3/XIST DKO"
)
new_panel <- make_heatmap_panel(
  new_color_index,
  new_breaks,
  c("WT 1", "WT 2", "KO 1", "KO 2", "KO+X1 1", "KO+X1 2"),
  "AFF3 + X1 drug"
)

tiff(
  output_file,
  width = figure_width,
  height = figure_height,
  units = "in",
  res = figure_dpi,
  compression = "lzw",
  type = "cairo"
)
grid.newpage()
pushViewport(
  viewport(
    layout = grid.layout(
      nrow = 1,
      ncol = 2,
      widths = unit(c(1, 1), "null")
    )
  )
)
pushViewport(viewport(layout.pos.row = 1, layout.pos.col = 1))
grid.draw(old_panel$gtable)
upViewport()
pushViewport(viewport(layout.pos.row = 1, layout.pos.col = 2))
grid.draw(new_panel$gtable)
upViewport(2)
dev.off()

stopifnot(file.exists(output_file), file.info(output_file)$size > 0)
file.info(output_file)[, c("size", "mtime")]

## Section boundaries used in the output

- Rows 1–6: autosomal neural rescue genes
- Rows 7–11: autosomal skeletal-muscle rescue genes